In [1]:
print("EDA")

EDA


In [2]:
import pandas as pd

df = pd.read_csv('reddit_politics_comments.csv')
df


,post_id,post_title,timestamp,platform,user_id,text,word_count,label
0,1rq7t5p,Leavitt Admits SAVE Act Will Make It Harder fo...,2026-03-10T20:06:21,reddit,AutoModerator,"**As a reminder, this subreddit [is for civil ...",141,NaN
1,1rq7t5p,Leavitt Admits SAVE Act Will Make It Harder fo...,2026-03-10T20:08:42,reddit,B-Z_B-S,It's a voter suppression bill. And it was writ...,23,NaN
2,1rq7t5p,Leavitt Admits SAVE Act Will Make It Harder fo...,2026-03-10T20:25:06,reddit,GWFfarley2k,Pretty sure that stopping women from voting wa...,25,NaN
3,1rq7t5p,Leavitt Admits SAVE Act Will Make It Harder fo...,2026-03-10T20:38:37,reddit,LuckyandBrownie,This is not meant to be a functional law. It’s...,27,NaN
4,1rq7t5p,Leavitt Admits SAVE Act Will Make It Harder fo...,2026-03-10T20:12:59,reddit,No_Somewhere_7109,Thune is unwilling to remove the Filibuster to...,152,NaN
...,...,...,...,...,...,...,...,...
1716,1rpw74x,Trump Goons Ordered to Hide Major Threat From ...,2026-03-10T15:13:21,reddit,Thebritishdovah,Probably because and I can't believe I am in f...,54,NaN
1717,1rpw74x,Trump Goons Ordered to Hide Major Threat From ...,2026-03-10T17:48:07,reddit,RogueAOV,I always wonder how a sleeper cell actually fu...,59,NaN
1718,1rpw74x,Trump Goons Ordered to Hide Major Threat From ...,2026-03-10T18:37:50,reddit,Bikewer,How would an even half-bright person NOT think...,30,NaN
1719,1rpw74x,Trump Goons Ordered to Hide Major Threat From ...,2026-03-10T22:34:38,reddit,InsomniaticWanderer,So we have no idea where Trump is?\n\nBecause ...,18,NaN


In [5]:
df['user_id'].value_counts()

user_id
AutoModerator           56
raiansar                15
B-Z_B-S                  8
Historical_Bend_2629     6
Memitim                  5
                        ..
Colonel-Mooseknuckle     1
Dimitri3p0               1
cybah                    1
bryce_lynch27            1
simmons777               1
Name: count, Length: 1471, dtype: int64

In [13]:
import re 

def clean_text(text):

    if pd.isna(text):
        return ""

    # lowercase
    text = text.lower()

    # remove newline
    text = text.replace("\n", " ")

    # remove url
    text = re.sub(r"http\S+", "", text)

    # remove special characters
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)

    # remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

def transform_data(input_path, output_path):

    df = pd.read_csv(input_path)

    # cleaning comment
    df["comment_clean"] = df["text"].apply(clean_text)

    # remove empty comment
    df = df[df["comment_clean"] != ""]

    # remove duplicate comment
    df = df.drop_duplicates(subset=["comment_clean"])

    df = df[
    (df["user_id"] != "AutoModerator") &
    (df["user_id"] != "[deleted]") &
    (df["user_id"] != "[removed]")
    ]

    # reset index
    df = df.reset_index(drop=True)

    # save cleaned data
    df.to_csv(output_path, index=False)

    print("Cleaning selesai")
    print("Jumlah data:", len(df))


In [14]:
transform_data(
    "reddit_politics_comments.csv",
    "reddit_comments_clean.csv"
)

Cleaning selesai
Jumlah data: 1665


In [15]:
df_cleaned = pd.read_csv('reddit_comments_clean.csv')
df_cleaned

,post_id,post_title,timestamp,platform,user_id,text,word_count,label,comment_clean
0,1rq7t5p,Leavitt Admits SAVE Act Will Make It Harder fo...,2026-03-10T20:08:42,reddit,B-Z_B-S,It's a voter suppression bill. And it was writ...,23,NaN,its a voter suppression bill and it was writte...
1,1rq7t5p,Leavitt Admits SAVE Act Will Make It Harder fo...,2026-03-10T20:25:06,reddit,GWFfarley2k,Pretty sure that stopping women from voting wa...,25,NaN,pretty sure that stopping women from voting wa...
2,1rq7t5p,Leavitt Admits SAVE Act Will Make It Harder fo...,2026-03-10T20:38:37,reddit,LuckyandBrownie,This is not meant to be a functional law. It’s...,27,NaN,this is not meant to be a functional law its m...
3,1rq7t5p,Leavitt Admits SAVE Act Will Make It Harder fo...,2026-03-10T20:12:59,reddit,No_Somewhere_7109,Thune is unwilling to remove the Filibuster to...,152,NaN,thune is unwilling to remove the filibuster to...
4,1rq7t5p,Leavitt Admits SAVE Act Will Make It Harder fo...,2026-03-10T20:36:47,reddit,PolicyWonka,Forcing tens of millions of Americans to go to...,38,NaN,forcing tens of millions of americans to go to...
...,...,...,...,...,...,...,...,...,...
1660,1rpw74x,Trump Goons Ordered to Hide Major Threat From ...,2026-03-10T15:13:21,reddit,Thebritishdovah,Probably because and I can't believe I am in f...,54,NaN,probably because and i cant believe i am in fa...
1661,1rpw74x,Trump Goons Ordered to Hide Major Threat From ...,2026-03-10T17:48:07,reddit,RogueAOV,I always wonder how a sleeper cell actually fu...,59,NaN,i always wonder how a sleeper cell actually fu...
1662,1rpw74x,Trump Goons Ordered to Hide Major Threat From ...,2026-03-10T18:37:50,reddit,Bikewer,How would an even half-bright person NOT think...,30,NaN,how would an even halfbright person not think ...
1663,1rpw74x,Trump Goons Ordered to Hide Major Threat From ...,2026-03-10T22:34:38,reddit,InsomniaticWanderer,So we have no idea where Trump is?\n\nBecause ...,18,NaN,so we have no idea where trump is because that...


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

texts = df_cleaned["text"]

# TF-IDF
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X = vectorizer.fit_transform(texts)

# KMeans clustering
kmeans = KMeans(
    n_clusters=3,
    random_state=42
)

df_cleaned["cluster"] = kmeans.fit_predict(X)
df_cleaned

,post_id,post_title,timestamp,platform,user_id,text,word_count,label,comment_clean,cluster
0,1rq7t5p,Leavitt Admits SAVE Act Will Make It Harder fo...,2026-03-10T20:08:42,reddit,B-Z_B-S,It's a voter suppression bill. And it was writ...,23,NaN,its a voter suppression bill and it was writte...,2
1,1rq7t5p,Leavitt Admits SAVE Act Will Make It Harder fo...,2026-03-10T20:25:06,reddit,GWFfarley2k,Pretty sure that stopping women from voting wa...,25,NaN,pretty sure that stopping women from voting wa...,2
2,1rq7t5p,Leavitt Admits SAVE Act Will Make It Harder fo...,2026-03-10T20:38:37,reddit,LuckyandBrownie,This is not meant to be a functional law. It’s...,27,NaN,this is not meant to be a functional law its m...,0
3,1rq7t5p,Leavitt Admits SAVE Act Will Make It Harder fo...,2026-03-10T20:12:59,reddit,No_Somewhere_7109,Thune is unwilling to remove the Filibuster to...,152,NaN,thune is unwilling to remove the filibuster to...,2
4,1rq7t5p,Leavitt Admits SAVE Act Will Make It Harder fo...,2026-03-10T20:36:47,reddit,PolicyWonka,Forcing tens of millions of Americans to go to...,38,NaN,forcing tens of millions of americans to go to...,2
...,...,...,...,...,...,...,...,...,...,...
1660,1rpw74x,Trump Goons Ordered to Hide Major Threat From ...,2026-03-10T15:13:21,reddit,Thebritishdovah,Probably because and I can't believe I am in f...,54,NaN,probably because and i cant believe i am in fa...,1
1661,1rpw74x,Trump Goons Ordered to Hide Major Threat From ...,2026-03-10T17:48:07,reddit,RogueAOV,I always wonder how a sleeper cell actually fu...,59,NaN,i always wonder how a sleeper cell actually fu...,1
1662,1rpw74x,Trump Goons Ordered to Hide Major Threat From ...,2026-03-10T18:37:50,reddit,Bikewer,How would an even half-bright person NOT think...,30,NaN,how would an even halfbright person not think ...,1
1663,1rpw74x,Trump Goons Ordered to Hide Major Threat From ...,2026-03-10T22:34:38,reddit,InsomniaticWanderer,So we have no idea where Trump is?\n\nBecause ...,18,NaN,so we have no idea where trump is because that...,2


In [22]:
for i, row in df_cleaned.head(20).iterrows():

    print("Comment :", row["text"])
    print("Label   :", row["cluster"])
    print("-"*50)

Comment : It's a voter suppression bill. And it was written to allow the Republican Party to win the midterms even with their extreme unpopularity.
Label   : 2
--------------------------------------------------
Comment : Pretty sure that stopping women from voting was another thing in Project2025. You know the one trump never heard of but is following almost exactly.
Label   : 2
--------------------------------------------------
Comment : This is not meant to be a functional law. It’s meant to throw the election into chaos with no one knowing what the fuck is going on.
Label   : 0
--------------------------------------------------
Comment : Thune is unwilling to remove the Filibuster to push it through, fortunately, and they don't have the votes besides. Trump may not be around long but most of the Senate will be and they know that there's no jamming the genie back in the bottle if the follow through - eventually the Democrats *will* get control again in some form and the Republicans w

In [23]:
df_cleaned['cluster'].value_counts()

cluster
2    1280
1     246
0     139
Name: count, dtype: int64